# 🔎 Cited Research Assistant: a web-search agent that answers with citations

**Track: Freestyle** · Built for the *AI Agents: Intensive Vibe Coding* capstone.

Ask a question. The agent **plans, calls a `web_search` tool (as many times as it needs), reads the results, and writes a clear answer with inline citations** `[1] [2]` plus a `Sources` list — so every factual claim is traceable.

**What this demonstrates (course concepts):**
- **Tool use** — a Python function exposed to the model as a callable tool.
- **An autonomous agent loop** — Gemini decides *when* and *how often* to search, then synthesises. We don't hard-code the steps.
- **Grounding & trust** — answers are tied to real retrieved sources, not the model's memory.
- **Vibe coding** — the whole agent is ~40 lines of Python driven by a natural-language system prompt.

## Setup (one minute)

1. **Turn on internet:** in the notebook sidebar → *Settings* → **Internet → On**.
2. **Add your Gemini API key as a Secret:** *Add-ons → Secrets* → add `GOOGLE_API_KEY`.
   Get a free key at **https://aistudio.google.com/apikey**.
3. Run the cells top to bottom.

In [ ]:
%pip install -q -U google-genai ddgs

In [ ]:
import os

# Load the Gemini API key from Kaggle Secrets (falls back to an env var for local runs)
try:
    from kaggle_secrets import UserSecretsClient
    GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
except Exception:
    GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY", "")

assert GOOGLE_API_KEY, "Add GOOGLE_API_KEY in Add-ons -> Secrets (or set the env var)."

from google import genai
from google.genai import types

client = genai.Client(api_key=GOOGLE_API_KEY)
MODEL = "gemini-2.5-flash"   # swap to "gemini-2.0-flash" if needed
print("Gemini client ready -", MODEL)

In [ ]:
# The agent's single tool: a web search.
try:
    from ddgs import DDGS                 # current package name
except ImportError:
    from duckduckgo_search import DDGS    # older name, identical API

def web_search(query: str) -> str:
    """Search the web for current information.
    Returns up to 5 results, each with a numbered title, URL and snippet.
    Call this whenever you need facts, evidence, or sources to cite."""
    print(f"   search: {query!r}")
    out = []
    try:
        with DDGS() as ddgs:
            for i, r in enumerate(ddgs.text(query, max_results=5), 1):
                out.append(f"[{i}] {r.get('title','')}\n    URL: {r.get('href','')}\n    {r.get('body','')}")
    except Exception as e:
        return f"search error: {e}"
    return "\n".join(out) if out else "No results found."

# quick smoke test
print(web_search("Mount Everest height meters")[:300])

## The agent

A system prompt defines the *behaviour*; the `web_search` function is the *tool*. Passing the function to Gemini turns on **automatic function calling** — the model runs its own loop: decide → search → read → (maybe search again) → answer. The `print` inside the tool lets us watch each step the agent takes.

In [ ]:
SYSTEM_INSTRUCTION = """You are a meticulous research assistant.

For every question that needs facts, follow this workflow:
1. Call `web_search` - as many times as needed, refining the query - to gather real sources.
2. Read the results, then write a clear, well-organised answer.
3. Support factual claims with inline citations like [1], [2] that map to the sources you actually retrieved.
4. End with a "Sources" section listing each number and its URL.

Never invent sources or URLs. If the evidence is thin or sources disagree, say so plainly."""

def ask(question: str, model: str = MODEL):
    print(f"Q: {question}\n--- the agent is working ---")
    resp = client.models.generate_content(
        model=model,
        contents=question,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_INSTRUCTION,
            tools=[web_search],     # -> automatic function-calling loop
            temperature=0.2,
        ),
    )
    # Show that the agent really used its tool (proof of the loop)
    afc = getattr(resp, "automatic_function_calling_history", None) or []
    n = sum(1 for c in afc for p in (getattr(c, "parts", None) or []) if getattr(p, "function_call", None))
    print("\n" + "="*72 + f"\nANSWER  (agent made {n} web_search call(s))\n" + "="*72)
    print(resp.text)
    return resp.text

## Try it

In [ ]:
_ = ask("What is the EU AI Act, and when do its main provisions take effect?")

In [ ]:
_ = ask("Compare the Sahara and Gobi deserts by area, location, and climate.")

In [ ]:
# Your turn - edit the question:
_ = ask("Who won the most recent FIFA World Cup, and who was the top scorer?")

## How it maps to the course — and where it could go next

**Agent technologies used**
- **Tool definition & calling** — `web_search` is registered as a tool; the model invokes it with arguments it chooses.
- **Autonomous loop** — the number of searches isn't scripted; the agent keeps going until it has enough to answer (visible in the `web_search call(s)` counter).
- **Grounded synthesis** — the final answer cites the retrieved sources, so claims are checkable rather than taken on trust.

**Honest limitations**
- Answer quality depends on search-result quality; an obscure query may return weak sources.
- The model maps citations to the sources it was given, but always sanity-check the linked URLs for anything important.

**Natural next steps**
- Add tools (a calculator, a PDF/URL reader, a date tool) and let the agent choose among them.
- Add a verifier pass that re-checks each citation actually supports its claim.
- Wrap `ask()` in a small chat UI for multi-turn follow-ups.

*Built conversationally (vibe coding) — a natural-language system prompt plus one tool is enough to make a useful, source-grounded agent.*